In [ ]:
%%spark

import os
import re
import sys
import traceback
from pprint import pformat
from typing import Dict, Optional
from pyspark.sql import DataFrame, SparkSession
from datetime import datetime
from pyspark.sql.functions import col, md5


class ErroPipeline(Exception):
    def __init__(
        self,
        codigo: str,
        mensagem: str,
        etapa: Optional[str] = None,
        objeto: Optional[str] = None,
        detalhes: Optional[dict] = None,
        acao: Optional[str] = None,
    ) -> None:
        super().__init__(mensagem)
        self.codigo = str(codigo).strip() or "ERRO_PIPELINE"
        self.mensagem = str(mensagem).strip()
        self.etapa = str(etapa).strip() if etapa else None
        self.objeto = str(objeto).strip() if objeto else None
        self.detalhes = dict(detalhes or {})
        self.acao = str(acao).strip() if acao else None

    def __str__(self) -> str:
        partes = [f"[{self.codigo}] {self.mensagem}"]
        if self.objeto:
            partes.append(f"objeto={self.objeto}")
        return " | ".join(partes)


class ErroAcessoDados(ErroPipeline):
    pass


class ErroContratoDados(ErroPipeline):
    pass


class ErroOperacional(ErroPipeline):
    pass


def _causa_raiz(exc):
    atual = exc
    visitados = set()

    while atual is not None and id(atual) not in visitados:
        visitados.add(id(atual))
        proxima = atual.__cause__ or atual.__context__
        if proxima is None:
            return atual
        atual = proxima

    return atual or exc


def _redigir_segredos(valor) -> str:
    texto = str(valor)
    texto = re.sub(
        r"(?i)\b(password|senha|secret|token|keytab)\s*([=:])\s*([^\s,;]+)",
        r"\1\2***",
        texto,
    )
    texto = re.sub(
        r"(?i)([a-z][a-z0-9+.-]*://)([^/\s:@]+):([^@/\s]+)@",
        r"\1***:***@",
        texto,
    )
    return texto


def _texto_log_resumido(valor, limite: int = 1000) -> str:
    texto = " ".join(_redigir_segredos(valor).split())
    if len(texto) <= limite:
        return texto
    return texto[:limite] + "..."


def _detalhes_log_seguros(detalhes: dict) -> str:
    palavras_sensiveis = ("PASSWORD", "SENHA", "SECRET", "TOKEN", "KEYTAB")
    partes = []

    for chave in sorted((detalhes or {}).keys(), key=str):
        chave_texto = str(chave)
        if any(palavra in chave_texto.upper() for palavra in palavras_sensiveis):
            valor = "***"
        else:
            valor = _texto_log_resumido(detalhes[chave])
        partes.append(f"{chave_texto}={valor}")

    return "; ".join(partes)

def publicar_tabelas_ando(
    df: DataFrame,
    database: str,
    tabela: str,
    modo: str,
    coluna_origem: str = "CD_CLI",
) -> None:

    try:
        if not database or not tabela:
            print("[ERRO] Database ou tabela não informados. Publicação não realizada.")
            return

        tbl_ando = f"{database}.ANDO_{tabela}"
        col_ando = f"DA_{coluna_origem}"

        if coluna_origem not in df.columns:
            print(
                f"[ALERTA] Coluna origem '{coluna_origem}' não existe. "
                f"Tabela '{tbl_ando}' NÃO será publicada."
            )
            return

        try:
            df_tmp = (
                df.withColumn(
                    col_ando,
                    md5(col(coluna_origem).cast("string"))
                )
                .drop(coluna_origem)
            )

            colunas_finais = [
                col_ando if c == coluna_origem else c
                for c in df.columns
            ]

            df_ando = df_tmp.select(*colunas_finais)

        except Exception as e:
            print(
                f"[ERRO] Falha ao gerar coluna '{col_ando}'. "
                f"Tabela '{tbl_ando}' NÃO publicada. Erro: {e}"
            )
            return

        try:
            (
                df_ando.write
                .mode(modo)
                .insertInto(tbl_ando)
            )

            print(
                f"[OK] Publicação concluída: '{tbl_ando}' "
                f"com '{col_ando}' (CHAR(32)) e sem exposição de "
                f"'{coluna_origem}'."
            )

        except Exception as e:
            print(f"[ERRO] Falha ao gravar tabela '{tbl_ando}': {e}")

    except Exception as e:
        print(f"[ERRO CRÍTICO] Falha geral na função: {e}")

def ler_variavel_ambiente_spark(nome_variavel: str) -> str:
    valor = os.environ.get(nome_variavel)

    if valor is None or not valor.strip():
        raise ErroOperacional(
            codigo="CONFIG_VARIAVEL_AUSENTE",
            mensagem="Variavel de ambiente obrigatoria nao informada.",
            objeto=nome_variavel,
            acao="Configurar a variavel no ambiente da sessao Spark antes de executar a rotina.",
        )

    return valor.strip()


def registrar_erro_rotina(etapa, exc):
    erro_classificado = isinstance(exc, ErroPipeline)
    causa = _causa_raiz(exc)
    etapa_final = exc.etapa if erro_classificado and exc.etapa else etapa
    codigo = exc.codigo if erro_classificado else "ERRO_NAO_CLASSIFICADO"
    mensagem = exc.mensagem if erro_classificado else str(exc)
    objeto = exc.objeto if erro_classificado else None
    detalhes = exc.detalhes if erro_classificado else {}
    acao = exc.acao if erro_classificado else None

    detalhes_seguros = _detalhes_log_seguros(detalhes)
    linhas = [
        "[PIPELINE][ERRO_FATAL]",
        f"etapa={_texto_log_resumido(etapa_final or '')}",
        f"codigo={codigo}",
        f"tipo={type(exc).__name__}",
        f"objeto={_texto_log_resumido(objeto or '')}",
        f"mensagem={_texto_log_resumido(mensagem)}",
        f"detalhes={detalhes_seguros}",
        "causa_raiz="
        f"{type(causa).__name__}: {_texto_log_resumido(causa)}",
        f"acao={_texto_log_resumido(acao or '')}",
    ]

    logger_rotina.error("\n".join(linhas))
    logger_rotina.error("[PIPELINE][TRACEBACK]\n" + _redigir_segredos(traceback.format_exc()))


def _use_logs(default=True) -> bool:
    value = os.environ.get("USE_LOGS")

    if value is None:
        return default

    return str(value).strip().lower() == "true"


def _flush() -> None:
    try:
        sys.stdout.flush()
    except Exception:
        pass

    try:
        sys.stderr.flush()
    except Exception:
        pass


def _print_log(msg: str) -> None:
    print(str(msg).replace("<br>", "\n"))
    _flush()


class ScreenLogger:
    def __init__(self, name: str = "PIPELINE") -> None:
        self.name = name

    def step(self, suffix: str) -> "ScreenLogger":
        return ScreenLogger(name=f"{self.name}.{suffix}")

    def _prefix(self) -> str:
        return f"[{self.name}] "

    def info(self, msg: str) -> None:
        if not _use_logs(True):
            return

        _print_log(f"{self._prefix()}{msg}")

    def error(self, msg: str) -> None:
        if not _use_logs(True):
            return

        _print_log(f"{self._prefix()}{msg}")

    def obj(self, value, title: str = None) -> None:
        if not _use_logs(True):
            return

        if title:
            _print_log(f"{self._prefix()}{title}")

        texto = (
            pformat(value, width=100, sort_dicts=False)
            if isinstance(value, (dict, list, tuple, set))
            else str(value)
        )
        _print_log(texto)

    def df(self, df, n: int = 20, truncate=True, title: str = None) -> None:
        if not _use_logs(True):
            return

        if title:
            _print_log(f"{self._prefix()}{title}")

        df.show(n=n, truncate=truncate)
        _flush()

    def df_schema(self, df, title: str = None) -> None:
        if not _use_logs(True):
            return

        if title:
            _print_log(f"{self._prefix()}{title}")

        df.printSchema()
        _flush()


class NullLogger:
    def step(self, suffix: str):
        return self

    def info(self, msg: str) -> None:
        pass

    def error(self, msg: str) -> None:
        pass

    def obj(self, value, title: str = None) -> None:
        pass

    def df(self, df, n: int = 20, truncate=True, title: str = None) -> None:
        pass

    def df_schema(self, df, title: str = None) -> None:
        pass


def criar_logger_spark(nome: str = "PIPELINE"):
    if _use_logs(True):
        return ScreenLogger(nome)

    return NullLogger()


logger = criar_logger_spark("PIPELINE")
logger_rotina = logger


class ClientOracleSpark:
    DEFAULT_DRIVER = "oracle.jdbc.OracleDriver"

    def __init__(self, spark: SparkSession, env: Optional[Dict[str, str]] = None) -> None:
        self.spark = spark
        self.env = env or dict(os.environ)

        def env_required(key: str) -> str:
            value = self.env.get(key)

            if value is None or str(value).strip() == "":
                raise ErroOperacional(
                    codigo="CONFIG_VARIAVEL_AUSENTE",
                    mensagem="Variavel Oracle obrigatoria nao informada.",
                    objeto=key,
                    acao="Configurar a variavel Oracle no ambiente da sessao Spark.",
                )

            return str(value).strip()

        def env_optional(key: str) -> Optional[str]:
            value = self.env.get(key)

            if value is None or str(value).strip() == "":
                return None

            return str(value).strip()

        self.user = env_required("VDP_ORACLE_USER")
        self.password = env_required("VDP_ORACLE_PASSWORD")
        self.host_1 = env_required("VDP_ORACLE_HOST_1")
        self.host_2 = env_optional("VDP_ORACLE_HOST_2")
        self.port = env_optional("VDP_ORACLE_PORTA") or "1521"
        self.service_name = env_optional("VDP_ORACLE_SERVICE_NAME") or env_required("VDP_ORACLE_SERVICE")
        self.schema = (env_optional("VDP_ORACLE_SCHEMA") or self.user).upper()
        self.driver = env_optional("VDP_ORACLE_DRIVER") or self.DEFAULT_DRIVER
        self.jar_path = env_optional("VDP_ORACLE_JAR") or "/dados/shared/bin/ojdbc8.jar"

        if self.driver != self.DEFAULT_DRIVER:
            raise ErroOperacional(
                codigo="CONFIG_DRIVER_INVALIDO",
                mensagem="Driver Oracle configurado e invalido.",
                objeto="VDP_ORACLE_DRIVER",
                detalhes={"driver_configurado": self.driver},
                acao=f"Configurar o driver Oracle esperado: {self.DEFAULT_DRIVER}.",
            )

        if self.host_2:
            self.url = (
                "jdbc:oracle:thin:@(DESCRIPTION="
                "(LOAD_BALANCE=OFF)"
                "(FAILOVER=ON)"
                "(CONNECT_TIMEOUT=10)"
                "(TRANSPORT_CONNECT_TIMEOUT=3)"
                "(RETRY_COUNT=3)"
                "(ADDRESS_LIST="
                f"(ADDRESS=(PROTOCOL=TCP)(HOST={self.host_1})(PORT={self.port}))"
                f"(ADDRESS=(PROTOCOL=TCP)(HOST={self.host_2})(PORT={self.port}))"
                ")"
                f"(CONNECT_DATA=(SERVICE_NAME={self.service_name}))"
                ")"
            )
        else:
            self.url = f"jdbc:oracle:thin:@//{self.host_1}:{self.port}/{self.service_name}"

    def run_select(
        self,
        sql: str,
        fetchsize: Optional[int] = None,
    ) -> DataFrame:
        query = (sql or "").strip()

        if not query:
            raise ValueError("sql nao pode ser vazio")

        if query.endswith(";"):
            query = query[:-1].strip()

        reader = (
            self.spark.read
            .format("jdbc")
            .option("url", self.url)
            .option("driver", self.driver)
            .option("user", self.user)
            .option("password", self.password)
            .option("dbtable", f"({query}) T")
        )

        if fetchsize is not None:
            if int(fetchsize) <= 0:
                raise ValueError("fetchsize deve ser maior que zero")

            reader = reader.option("fetchsize", int(fetchsize))

        try:
            return reader.load()
        except ErroPipeline:
            raise
        except Exception as exc:
            raise ErroAcessoDados(
                codigo="ORACLE_LEITURA_FALHOU",
                mensagem="Falha ao executar leitura Oracle.",
                objeto=f"ORACLE:{self.schema}",
                detalhes={"operacao": "SELECT"},
                acao="Verificar conexao, permissao, disponibilidade e objeto consultado no Oracle.",
            ) from exc

    def selecionar_tabela(
            self,
            nome_tabela: str,
            owner: Optional[str] = None,
            fetchsize: Optional[int] = None,
            show: bool = False,
            truncate: bool = True,
            n: int = 20,
        ) -> DataFrame:
            owner_final = (owner or self.schema or "").strip().upper()
            tabela_final = (nome_tabela or "").strip().upper()

            if not owner_final:
                raise ValueError("owner/schema nao pode ser vazio")

            if not tabela_final:
                raise ValueError("nome_tabela nao pode ser vazio")

            if "." in tabela_final:
                raise ValueError(
                    "nome_tabela deve receber apenas o nome da tabela. "
                    "Informe owner/schema separadamente."
                )

            if int(n) <= 0:
                raise ValueError("n deve ser maior que zero")

            df = self.run_select(
                sql=f"SELECT * FROM {owner_final}.{tabela_final}",
                fetchsize=fetchsize,
            )

            if show:
                df.show(n=int(n), truncate=truncate)

            return df

    def execute(self, sql: str) -> None:
            command = (sql or "").strip()

            if not command:
                raise ValueError("sql nao pode ser vazio")

            if command.endswith(";"):
                command = command[:-1].strip()

            jvm = self.spark._jvm
            conn = None
            stmt = None

            def criar_conexao_driver_manager():
                try:
                    jvm.java.lang.Class.forName(self.driver)
                except Exception:
                    context_loader = (
                        jvm.java.lang.Thread
                        .currentThread()
                        .getContextClassLoader()
                    )

                    jvm.java.lang.Class.forName(
                        self.driver,
                        True,
                        context_loader
                    )

                return jvm.java.sql.DriverManager.getConnection(
                    self.url,
                    self.user,
                    self.password,
                )

            def criar_conexao_url_classloader():
                gateway = self.spark.sparkContext._gateway

                jar_file = jvm.java.io.File(self.jar_path)

                if not jar_file.exists():
                    raise ValueError(
                        f"Jar Oracle nao encontrado em: {self.jar_path}. "
                        "Informe VDP_ORACLE_JAR ou carregue o ojdbc8.jar na sessao Spark."
                    )

                jar_url = jar_file.toURI().toURL()

                urls = gateway.new_array(jvm.java.net.URL, 1)
                urls[0] = jar_url

                parent_loader = (
                    jvm.java.lang.Thread
                    .currentThread()
                    .getContextClassLoader()
                )

                loader = jvm.java.net.URLClassLoader(urls, parent_loader)

                driver_class = jvm.java.lang.Class.forName(
                    self.driver,
                    True,
                    loader
                )

                driver = driver_class.newInstance()

                props = jvm.java.util.Properties()
                props.setProperty("user", self.user)
                props.setProperty("password", self.password)

                return driver.connect(self.url, props)

            try:
                try:
                    conn = criar_conexao_driver_manager()
                except Exception:
                    conn = criar_conexao_url_classloader()

                conn.setAutoCommit(False)

                stmt = conn.createStatement()
                stmt.execute(command)

                conn.commit()

            except ErroPipeline:
                if conn is not None:
                    try:
                        conn.rollback()
                    except Exception:
                        pass

                raise

            except Exception as exc:
                if conn is not None:
                    try:
                        conn.rollback()
                    except Exception:
                        pass

                raise ErroAcessoDados(
                    codigo="ORACLE_COMANDO_FALHOU",
                    mensagem="Falha ao executar comando Oracle.",
                    objeto=f"ORACLE:{self.schema}",
                    detalhes={"operacao": "COMANDO_SEM_RETORNO"},
                    acao="Verificar conexao, permissao, disponibilidade e comando executado no Oracle.",
                ) from exc

            finally:
                if stmt is not None:
                    try:
                        stmt.close()
                    except Exception:
                        pass

                if conn is not None:
                    try:
                        conn.close()
                    except Exception:
                        pass

            return None

    def carregar_df(
            self,
            df: DataFrame,
            table_name: str,
            owner: Optional[str] = None,
            batchsize: int = 5000,
            num_partitions: int = 1,
        ) -> None:
            if df is None:
                raise ValueError("df nao pode ser None")

            if int(batchsize) <= 0:
                raise ValueError("batchsize deve ser maior que zero")

            if int(num_partitions) <= 0:
                raise ValueError("num_partitions deve ser maior que zero")

            table = (table_name or "").strip()

            if not table:
                raise ValueError("table_name nao pode ser vazio")

            if "." in table:
                raise ValueError(
                    "table_name deve receber apenas o nome da tabela. "
                    "Informe owner/schema separadamente."
                )

            final_owner = (owner or self.schema or "").strip()

            if not final_owner:
                raise ValueError("owner/schema nao pode ser vazio")

            full_table_name = f"{final_owner.upper()}.{table.upper()}"

            writer_df = df.coalesce(int(num_partitions))

            try:
                (
                    writer_df.write
                    .format("jdbc")
                    .mode("append")
                    .option("url", self.url)
                    .option("driver", self.driver)
                    .option("user", self.user)
                    .option("password", self.password)
                    .option("dbtable", full_table_name)
                    .option("batchsize", int(batchsize))
                    .save()
                )
            except ErroPipeline:
                raise
            except Exception as exc:
                raise ErroAcessoDados(
                    codigo="ORACLE_ESCRITA_FALHOU",
                    mensagem="Falha ao gravar DataFrame no Oracle.",
                    objeto=full_table_name,
                    detalhes={"operacao": "APPEND"},
                    acao="Verificar conexao, permissao, disponibilidade e limites fisicos da tabela Oracle.",
                ) from exc

            return None

    def limpar_tabela(
            self,
            table_name: str,
            owner: Optional[str] = None,
            use_truncate: bool = False,
            batchsize: int = 5000,
        ) -> None:
            if int(batchsize) <= 0:
                raise ValueError("batchsize deve ser maior que zero")
            
            table = (table_name or "").strip()

            if not table:
                raise ValueError("table_name nao pode ser vazio")

            if "." in table:
                raise ValueError(
                    "table_name deve receber apenas o nome da tabela. "
                    "Informe owner/schema separadamente."
                )

            final_owner = (owner or self.schema or "").strip()

            if not final_owner:
                raise ValueError("owner/schema nao pode ser vazio")

            full_table_name = f"{final_owner.upper()}.{table.upper()}"

            if use_truncate:
                self.execute(f"TRUNCATE TABLE {full_table_name}")
                return None

            lote = int(batchsize)

            while True:
                df_count = self.run_select(
                    f"SELECT COUNT(1) AS QTD FROM {full_table_name}"
                )

                qtd_restante = int(df_count.collect()[0]["QTD"])

                if qtd_restante == 0:
                    break

                self.execute(
                    f"DELETE FROM {full_table_name} WHERE ROWNUM <= {lote}"
                )

            return None

    def reload_dataframe(
            self,
            df: DataFrame,
            table_name: str,
            owner: Optional[str] = None,
            batchsize: int = 5000,
            num_partitions: int = 1,
            use_truncate: bool = False,
        ) -> None:
            if df is None:
                raise ValueError("df nao pode ser None")

            table = (table_name or "").strip()

            if not table:
                raise ValueError("table_name nao pode ser vazio")

            if "." in table:
                raise ValueError(
                    "table_name deve receber apenas o nome da tabela. "
                    "Informe owner/schema separadamente."
                )

            final_owner = (owner or self.schema or "").strip()

            if not final_owner:
                raise ValueError("owner/schema nao pode ser vazio")

            self.limpar_tabela(
                table_name=table,
                owner=final_owner,
                use_truncate=use_truncate,
                batchsize=batchsize,
            )

            self.carregar_df(
                df=df,
                table_name=table,
                owner=final_owner,
                batchsize=batchsize,
                num_partitions=num_partitions,
            )

            return None

def criar_cliente_oracle_spark(env: Optional[Dict[str, str]] = None) -> ClientOracleSpark:
    return ClientOracleSpark(spark=spark, env=env)


class ClientDb2Spark:
    DEFAULT_DRIVER = "com.ibm.db2.jcc.DB2Driver"

    def __init__(self, spark: SparkSession, env: Optional[Dict[str, str]] = None) -> None:
        self.spark = spark
        self.env = env or dict(os.environ)

        def env_required(key: str) -> str:
            value = self.env.get(key)

            if value is None or str(value).strip() == "":
                raise ErroOperacional(
                    codigo="CONFIG_VARIAVEL_AUSENTE",
                    mensagem="Variavel DB2 obrigatoria nao informada.",
                    objeto=key,
                    acao="Configurar a variavel DB2 no ambiente da sessao Spark.",
                )

            return str(value).strip()

        def env_optional(key: str) -> Optional[str]:
            value = self.env.get(key)

            if value is None or str(value).strip() == "":
                return None

            return str(value).strip()

        self.user = env_required("DB2_USER")
        self.password = env_required("DB2_PASSWORD")
        self.host = env_required("DB2_HOST")
        self.port = env_optional("DB2_PORTA") or env_optional("VDP_DB2_PORTA") or "50100"
        self.database = env_required("DB2_DATABASE")
        self.driver = env_optional("DB2_DRIVER") or self.DEFAULT_DRIVER

        if self.driver != self.DEFAULT_DRIVER:
            raise ErroOperacional(
                codigo="CONFIG_DRIVER_INVALIDO",
                mensagem="Driver DB2 configurado e invalido.",
                objeto="DB2_DRIVER",
                detalhes={"driver_configurado": self.driver},
                acao=f"Configurar o driver DB2 esperado: {self.DEFAULT_DRIVER}.",
            )

        self.url = f"jdbc:db2://{self.host}:{self.port}/{self.database}"

    def run_select(
        self,
        sql: str,
        fetchsize: Optional[int] = None,
        partition_column: Optional[str] = None,
        lower_bound: Optional[int] = None,
        upper_bound: Optional[int] = None,
        num_partitions: Optional[int] = None,
        query_timeout: Optional[int] = None,
    ) -> DataFrame:
        query = (sql or "").strip()

        if not query:
            raise ValueError("sql nao pode ser vazio")

        if query.endswith(";"):
            query = query[:-1].strip()

        reader = (
            self.spark.read
            .format("jdbc")
            .option("url", self.url)
            .option("driver", self.driver)
            .option("user", self.user)
            .option("password", self.password)
            .option("dbtable", f"({query}) T")
        )

        if fetchsize is not None:
            if int(fetchsize) <= 0:
                raise ValueError("fetchsize deve ser maior que zero")

            reader = reader.option("fetchsize", int(fetchsize))

        if query_timeout is not None:
            if int(query_timeout) <= 0:
                raise ValueError("query_timeout deve ser maior que zero")

            reader = reader.option("queryTimeout", int(query_timeout))

        parametros_particao = [
            partition_column,
            lower_bound,
            upper_bound,
            num_partitions,
        ]

        if any(valor is not None for valor in parametros_particao):
            if any(valor is None for valor in parametros_particao):
                raise ValueError(
                    "Para leitura particionada, informe partition_column, "
                    "lower_bound, upper_bound e num_partitions."
                )

            partition_column = (partition_column or "").strip()
            lower_bound_final = int(lower_bound)
            upper_bound_final = int(upper_bound)
            num_partitions_final = int(num_partitions)

            if not partition_column:
                raise ValueError("partition_column nao pode ser vazio")

            if lower_bound_final >= upper_bound_final:
                raise ValueError("lower_bound deve ser menor que upper_bound")

            if num_partitions_final <= 0:
                raise ValueError("num_partitions deve ser maior que zero")

            reader = (
                reader
                .option("partitionColumn", partition_column)
                .option("lowerBound", lower_bound_final)
                .option("upperBound", upper_bound_final)
                .option("numPartitions", num_partitions_final)
            )

        try:
            return reader.load()
        except ErroPipeline:
            raise
        except Exception as exc:
            raise ErroAcessoDados(
                codigo="DB2_LEITURA_FALHOU",
                mensagem="Falha ao executar leitura DB2.",
                objeto=f"DB2:{self.database}",
                detalhes={"operacao": "SELECT"},
                acao="Verificar conexao, permissao, disponibilidade e objeto consultado no DB2.",
            ) from exc

    def diagnosticar_tabela(
        self,
        schema: str,
        nome_tabela: str,
        data_inicio=None,
        show: bool = True,
        truncate: bool = False,
        limite_sem_filtro: int = 250_000,
        max_linhas_recorte: int = 200_000,
        limite_valores_unicos: int = 100,
        max_janela_automatica_dias: int = 90,
        exigir_indice_em_tabela_grande: bool = True,
    ) -> dict:
        """
        Diagnóstico automático, rápido e conservador de tabela DB2.

        Objetivos
        ---------
        1. Conhecer toda a estrutura da tabela via catálogo.
        2. Identificar automaticamente:
        - colunas;
        - tipos;
        - naturezas;
        - datas/timestamps;
        - índices;
        - possíveis colunas de filtro;
        - possíveis colunas de particionamento.
        3. Evitar varredura desnecessária em tabelas muito grandes.
        4. Escolher automaticamente uma coluna temporal para recorte,
        quando necessário.
        5. Usar somente um recorte controlado dos dados.
        6. Exibir valores únicos somente para colunas com perfil de
        domínio pequeno.
        7. Não executar DISTINCT para colunas conhecidamente de alta
        cardinalidade.
        8. Continuar entregando os principais insumos para
        selecionar_tabela(...).

        Parâmetros
        ----------
        schema:
            Schema DB2.

        nome_tabela:
            Nome da tabela ou view.

        data_inicio:
            Data mínima desejada para o diagnóstico dos dados.
            Pode ser:
            - "YYYY-MM-DD";
            - datetime.date;
            - datetime.datetime;
            - None.

            A coluna usada para esse filtro é descoberta automaticamente.

            Em tabela muito grande, se o período solicitado ainda produzir
            um universo excessivo, a função pode avançar automaticamente
            a data efetiva do recorte.

            A alteração nunca é silenciosa: a data solicitada e a data
            efetivamente utilizada são retornadas.

        limite_sem_filtro:
            Até essa estimativa de linhas, a tabela pode ser diagnosticada
            sem exigir filtro temporal.

        max_linhas_recorte:
            Quantidade máxima de linhas materializadas para o diagnóstico
            de conteúdo.

        limite_valores_unicos:
            Máximo de valores diferentes que caracteriza um domínio pequeno
            que vale a pena exibir.

        max_janela_automatica_dias:
            Se data_inicio não for informada e a tabela for grande, procura
            um recorte recente de até esta quantidade de dias.

        exigir_indice_em_tabela_grande:
            Se True, tabelas grandes somente utilizam automaticamente uma
            coluna temporal que seja líder de algum índice.

        Retorno
        -------
        dict contendo:
            - metadados da tabela;
            - metadados das colunas;
            - estratégia de recorte;
            - coluna temporal escolhida;
            - valores únicos observados de domínios pequenos;
            - colunas de data;
            - candidatas a particionamento;
            - bounds seguros;
            - fetchsize;
            - num_partitions;
            - alertas.
        """

        from datetime import date, datetime, time, timedelta

        from pyspark.sql import functions as F

        # ================================================================
        # Validações básicas
        # ================================================================

        if max_linhas_recorte <= 0:
            raise ValueError(
                "max_linhas_recorte deve ser maior que zero."
            )

        if limite_sem_filtro <= 0:
            raise ValueError(
                "limite_sem_filtro deve ser maior que zero."
            )

        if limite_valores_unicos <= 0:
            raise ValueError(
                "limite_valores_unicos deve ser maior que zero."
            )

        if max_janela_automatica_dias <= 0:
            raise ValueError(
                "max_janela_automatica_dias deve ser maior que zero."
            )

        schema_final = self._normalizar_identificador(
            schema,
            "schema",
        )

        tabela_final = self._normalizar_identificador(
            nome_tabela,
            "nome_tabela",
        )

        nome_completo = f"{schema_final}.{tabela_final}"

        # ================================================================
        # Naturezas corporativas
        # ================================================================

        naturezas = {
            "AA": "ANO",
            "BL": "BLOB",
            "CD": "CODIGO",
            "DA": "DADO_ANONIMIZADO",
            "DT": "DATA",
            "DD": "DIA",
            "DV": "DIGITO_VERIFICADOR",
            "HH": "HORA",
            "HR": "HORARIO",
            "IM": "IMAGEM",
            "IN": "INDICADOR",
            "IC": "INDICE",
            "JS": "JSON",
            "LS": "LISTA",
            "MM": "MES",
            "NM": "NOME",
            "NR": "NUMERO",
            "OG": "OBJETO_GEOESPACIAL",
            "PC": "PERCENTUAL",
            "QT": "QUANTIDADE",
            "RI": "ROWID",
            "SG": "SIGLA",
            "SB": "SIMBOLO",
            "SM": "SOM",
            "TX": "TEXTO",
            "TS": "TIMESTAMP",
            "VL": "VALOR",
        }

        tipos_data = {
            "DATE",
            "TIMESTAMP",
            "TIMESTMP",
        }

        tipos_numericos = {
            "SMALLINT",
            "INTEGER",
            "BIGINT",
            "DECIMAL",
            "NUMERIC",
            "DECFLOAT",
            "REAL",
            "FLOAT",
            "DOUBLE",
        }

        tipos_lob = {
            "BLOB",
            "CLOB",
            "DBCLOB",
            "XML",
            "LONGVAR",
            "LONGVARG",
            "LONGVARB",
        }

        tipos_textuais = {
            "CHAR",
            "VARCHAR",
            "GRAPHIC",
            "VARGRAPH",
        }

        tipos_particao = tipos_data | tipos_numericos

        naturezas_dominio = {
            "IN",
            "CD",
            "SG",
            "SB",
            "DV",
            "AA",
            "MM",
            "DD",
        }

        alertas = []

        # ================================================================
        # Helpers internos
        # ================================================================

        def _row_dict(row):
            if hasattr(row, "asDict"):
                return row.asDict(recursive=True)
            return dict(row)

        def _float_seguro(valor, default=-1):
            try:
                if valor is None:
                    return default
                return float(valor)
            except (TypeError, ValueError):
                return default

        def _int_seguro(valor, default=None):
            try:
                if valor is None:
                    return default
                return int(valor)
            except (TypeError, ValueError):
                return default

        def _natureza_coluna(nome_coluna):
            prefixo = (
                str(nome_coluna)
                .strip()
                .upper()
                .split("_", 1)[0]
            )

            return (
                prefixo,
                naturezas.get(prefixo, "NAO_IDENTIFICADA"),
            )

        def _normalizar_datetime(valor):
            if valor is None:
                return None

            if isinstance(valor, datetime):
                return valor

            if isinstance(valor, date):
                return datetime.combine(
                    valor,
                    time.min,
                )

            texto = str(valor).strip()

            tentativas = [
                texto,
                texto.replace(" ", "T"),
            ]

            for tentativa in tentativas:
                try:
                    return datetime.fromisoformat(tentativa)
                except ValueError:
                    pass

            try:
                data = date.fromisoformat(texto[:10])

                return datetime.combine(
                    data,
                    time.min,
                )
            except ValueError as exc:
                raise ValueError(
                    f"Valor temporal inválido: {valor}"
                ) from exc

        def _literal_temporal(valor, tipo_db2):
            dt = _normalizar_datetime(valor)

            if tipo_db2 == "DATE":
                texto = dt.date().isoformat()

                return (
                    f"CAST('{texto}' AS DATE)"
                )

            texto = dt.strftime(
                "%Y-%m-%d %H:%M:%S.%f"
            )

            return (
                f"CAST('{texto}' AS TIMESTAMP)"
            )

        def _extrair_primeiro_valor(rows, nome):
            if not rows:
                return None

            dados = _row_dict(rows[0])

            return dados.get(nome)

        # ================================================================
        # 1. Metadados da tabela
        # ================================================================

        sql_tabela = f"""
            SELECT
                TYPE AS TIPO_OBJETO,
                CARDF AS ESTIMATIVA_LINHAS
            FROM SYSIBM.SYSTABLES
            WHERE CREATOR = '{schema_final}'
            AND NAME = '{tabela_final}'
        """

        linhas_tabela = (
            self.run_select(sql_tabela)
            .collect()
        )

        if not linhas_tabela:
            raise ValueError(
                f"Tabela ou view não encontrada: "
                f"{nome_completo}"
            )

        dados_tabela = _row_dict(
            linhas_tabela[0]
        )

        tipo_objeto = str(
            dados_tabela.get("TIPO_OBJETO") or ""
        ).strip()

        estimativa_linhas = _float_seguro(
            dados_tabela.get("ESTIMATIVA_LINHAS")
        )

        # ================================================================
        # 2. Metadados das colunas
        #
        # COLCARD é utilizado apenas como proteção interna.
        # Se o catálogo não suportar ou não possuir a informação,
        # seguimos sem ela.
        # ================================================================

        sql_colunas_com_cardinalidade = f"""
            SELECT
                NAME AS COLUNA,
                COLNO AS POSICAO,
                COLTYPE AS TIPO_DB2,
                LENGTH AS TAMANHO,
                NULLS AS ACEITA_NULO,
                COLCARD AS CARDINALIDADE_CATALOGO
            FROM SYSIBM.SYSCOLUMNS
            WHERE TBCREATOR = '{schema_final}'
            AND TBNAME = '{tabela_final}'
            ORDER BY COLNO
        """

        try:
            colunas_rows = (
                self.run_select(
                    sql_colunas_com_cardinalidade
                )
                .collect()
            )

        except Exception:
            sql_colunas_sem_cardinalidade = f"""
                SELECT
                    NAME AS COLUNA,
                    COLNO AS POSICAO,
                    COLTYPE AS TIPO_DB2,
                    LENGTH AS TAMANHO,
                    NULLS AS ACEITA_NULO
                FROM SYSIBM.SYSCOLUMNS
                WHERE TBCREATOR = '{schema_final}'
                AND TBNAME = '{tabela_final}'
                ORDER BY COLNO
            """

            colunas_rows = (
                self.run_select(
                    sql_colunas_sem_cardinalidade
                )
                .collect()
            )

            alertas.append(
                "CARDINALIDADE_CATALOGO_INDISPONIVEL"
            )

        if not colunas_rows:
            raise ValueError(
                f"Nenhuma coluna encontrada para "
                f"{nome_completo}"
            )

        # ================================================================
        # 3. Índices
        # ================================================================

        sql_indices = f"""
            SELECT
                I.NAME AS INDICE,
                K.COLNAME AS COLUNA,
                K.COLSEQ AS POSICAO_INDICE
            FROM SYSIBM.SYSINDEXES I
            INNER JOIN SYSIBM.SYSKEYS K
                ON I.CREATOR = K.IXCREATOR
            AND I.NAME = K.IXNAME
            WHERE I.TBCREATOR = '{schema_final}'
            AND I.TBNAME = '{tabela_final}'
            ORDER BY
                I.NAME,
                K.COLSEQ
        """

        try:
            indices_rows = (
                self.run_select(sql_indices)
                .collect()
            )
        except Exception:
            indices_rows = []

            alertas.append(
                "METADADOS_DE_INDICE_INDISPONIVEIS"
            )

        colunas_indice_lider = set()
        indices_por_coluna = {}

        for linha in indices_rows:
            dados = _row_dict(linha)

            nome_coluna = str(
                dados.get("COLUNA") or ""
            ).strip().upper()

            nome_indice = str(
                dados.get("INDICE") or ""
            ).strip().upper()

            posicao = _int_seguro(
                dados.get("POSICAO_INDICE")
            )

            if not nome_coluna:
                continue

            indices_por_coluna.setdefault(
                nome_coluna,
                [],
            ).append({
                "indice": nome_indice,
                "posicao": posicao,
            })

            if posicao == 1:
                colunas_indice_lider.add(
                    nome_coluna
                )

        # ================================================================
        # 4. Montagem do catálogo interno das colunas
        # ================================================================

        filtrar_cols = []
        col_data = []
        col_particao = []

        tamanho_linha_estimado = 0
        possui_lob = False

        colunas_info = []

        for linha in colunas_rows:
            dados = _row_dict(linha)

            nome_coluna = str(
                dados.get("COLUNA") or ""
            ).strip().upper()

            tipo_coluna = str(
                dados.get("TIPO_DB2") or ""
            ).strip().upper()

            tamanho = _int_seguro(
                dados.get("TAMANHO"),
                default=0,
            )

            aceita_nulo = str(
                dados.get("ACEITA_NULO") or ""
            ).strip().upper()

            cardinalidade_catalogo = _float_seguro(
                dados.get(
                    "CARDINALIDADE_CATALOGO"
                ),
                default=-1,
            )

            prefixo, natureza = (
                _natureza_coluna(nome_coluna)
            )

            indice_lider = (
                nome_coluna
                in colunas_indice_lider
            )

            filtrar_cols.append(
                nome_coluna
            )

            if tipo_coluna in tipos_data:
                col_data.append(
                    nome_coluna
                )

            if tipo_coluna in tipos_lob:
                possui_lob = True

            try:
                tamanho_linha_estimado += (
                    self._estimar_tamanho_coluna(
                        tipo=tipo_coluna,
                        tamanho=tamanho,
                    )
                )
            except Exception:
                tamanho_linha_estimado += max(
                    0,
                    tamanho,
                )

            if (
                tipo_coluna in tipos_particao
                and aceita_nulo == "N"
                and indice_lider
            ):
                col_particao.append(
                    nome_coluna
                )

            colunas_info.append({
                "coluna": nome_coluna,
                "posicao": _int_seguro(
                    dados.get("POSICAO")
                ),
                "tipo_db2": tipo_coluna,
                "tamanho": tamanho,
                "aceita_nulo": aceita_nulo,
                "prefixo_natureza": prefixo,
                "natureza": natureza,
                "indice_lider": indice_lider,
                "indices": indices_por_coluna.get(
                    nome_coluna,
                    [],
                ),
                "cardinalidade_catalogo": (
                    cardinalidade_catalogo
                    if cardinalidade_catalogo >= 0
                    else None
                ),
                "diagnostico_valores": (
                    "NAO_ANALISADO"
                ),
                "valores_unicos": None,
            })

        coluna_por_nome = {
            item["coluna"]: item
            for item in colunas_info
        }

        # ================================================================
        # 5. Identificação automática das melhores datas
        # ================================================================

        palavras_negocio = {
            "TRAN",
            "MOV",
            "EVT",
            "LCTO",
            "OPER",
            "FAT",
            "REF",
        }

        palavras_tecnicas = {
            "ATL",
            "ATZ",
            "INCL",
            "CRI",
            "CARG",
            "PROC",
            "PRCS",
            "ING",
            "EXTR",
        }

        candidatas_temporais = []

        for item in colunas_info:
            if item["tipo_db2"] not in tipos_data:
                continue

            nome = item["coluna"]

            tokens = set(
                nome.split("_")
            )

            score = 100
            motivos = [
                "TIPO_TEMPORAL",
            ]

            if item["prefixo_natureza"] == "DT":
                score += 50
                motivos.append(
                    "NATUREZA_DATA"
                )

            elif item["prefixo_natureza"] == "TS":
                score += 20
                motivos.append(
                    "NATUREZA_TIMESTAMP"
                )

            if item["indice_lider"]:
                score += 120
                motivos.append(
                    "LIDER_DE_INDICE"
                )

            if item["aceita_nulo"] == "N":
                score += 15
                motivos.append(
                    "NOT_NULL"
                )

            if tokens & palavras_negocio:
                score += 35
                motivos.append(
                    "APARENCIA_DATA_NEGOCIO"
                )

            if tokens & palavras_tecnicas:
                score -= 40
                motivos.append(
                    "APARENCIA_DATA_TECNICA"
                )

            candidatas_temporais.append({
                "coluna": nome,
                "tipo_db2": item["tipo_db2"],
                "indice_lider": item[
                    "indice_lider"
                ],
                "score": score,
                "motivos": motivos,
            })

        candidatas_temporais.sort(
            key=lambda x: (
                x["score"],
                x["indice_lider"],
            ),
            reverse=True,
        )

        # ================================================================
        # 6. Tamanho lógico da tabela
        # ================================================================

        tabela_estimativa_desconhecida = (
            estimativa_linhas < 0
        )

        tabela_grande = (
            tabela_estimativa_desconhecida
            or estimativa_linhas
            > limite_sem_filtro
        )

        # ================================================================
        # 7. Escolha automática da coluna de corte
        # ================================================================

        coluna_filtro = None
        tipo_coluna_filtro = None

        candidatas_seguras = (
            candidatas_temporais
        )

        if (
            tabela_grande
            and exigir_indice_em_tabela_grande
        ):
            candidatas_seguras = [
                item
                for item in candidatas_temporais
                if item["indice_lider"]
            ]

        if candidatas_seguras:
            coluna_filtro = (
                candidatas_seguras[0]["coluna"]
            )

            tipo_coluna_filtro = (
                candidatas_seguras[0][
                    "tipo_db2"
                ]
            )

        # ================================================================
        # 8. Data máxima da coluna escolhida
        #
        # ORDER BY ... DESC + FETCH FIRST 1:
        # preferível a MAX() para aproveitar índice líder quando houver.
        # ================================================================

        data_maxima = None

        if coluna_filtro is not None:
            sql_data_maxima = f"""
                SELECT
                    {coluna_filtro} AS VALOR_MAXIMO
                FROM {nome_completo}
                WHERE {coluna_filtro} IS NOT NULL
                ORDER BY {coluna_filtro} DESC
                FETCH FIRST 1 ROW ONLY
            """

            try:
                linhas_max = (
                    self.run_select(
                        sql_data_maxima
                    )
                    .collect()
                )

                valor_max = _extrair_primeiro_valor(
                    linhas_max,
                    "VALOR_MAXIMO",
                )

                data_maxima = (
                    _normalizar_datetime(
                        valor_max
                    )
                    if valor_max is not None
                    else None
                )

            except Exception:
                alertas.append(
                    "NAO_FOI_POSSIVEL_OBTER_DATA_MAXIMA"
                )

                coluna_filtro = None
                tipo_coluna_filtro = None

        # ================================================================
        # 9. Função de contagem limitada
        #
        # Não faz COUNT(*) do universo inteiro.
        #
        # O DB2 para quando encontra max_linhas_recorte + 1.
        # ================================================================

        def _contagem_limitada(
            coluna,
            tipo_db2,
            inicio,
            fim,
        ):
            limite = (
                max_linhas_recorte + 1
            )

            inicio_sql = (
                _literal_temporal(
                    inicio,
                    tipo_db2,
                )
            )

            fim_sql = (
                _literal_temporal(
                    fim,
                    tipo_db2,
                )
            )

            sql = f"""
                SELECT
                    COUNT(*) AS QTD
                FROM (
                    SELECT
                        1 AS X
                    FROM {nome_completo}
                    WHERE {coluna} >= {inicio_sql}
                    AND {coluna} <= {fim_sql}
                    FETCH FIRST {limite} ROWS ONLY
                ) T
            """

            linhas = (
                self.run_select(sql)
                .collect()
            )

            valor = _extrair_primeiro_valor(
                linhas,
                "QTD",
            )

            return _int_seguro(
                valor,
                default=0,
            )

        # ================================================================
        # 10. Encontrar automaticamente o início seguro
        #
        # Usa busca binária por DIA.
        #
        # Não tenta estimar distribuição.
        # Apenas responde:
        #
        # "este intervalo cabe ou não no orçamento?"
        # ================================================================

        def _buscar_inicio_seguro(
            inicio_minimo,
            fim,
        ):
            qtd_inicial = _contagem_limitada(
                coluna=coluna_filtro,
                tipo_db2=tipo_coluna_filtro,
                inicio=inicio_minimo,
                fim=fim,
            )

            if qtd_inicial <= max_linhas_recorte:
                return (
                    inicio_minimo,
                    qtd_inicial,
                    False,
                )

            dia_inicio = (
                inicio_minimo.date()
            )

            dia_fim = fim.date()

            melhor_inicio = datetime.combine(
                dia_fim,
                time.min,
            )

            melhor_qtd = None

            while dia_inicio <= dia_fim:
                distancia = (
                    dia_fim - dia_inicio
                ).days

                dia_meio = (
                    dia_inicio
                    + timedelta(
                        days=distancia // 2
                    )
                )

                inicio_teste = datetime.combine(
                    dia_meio,
                    time.min,
                )

                qtd = _contagem_limitada(
                    coluna=coluna_filtro,
                    tipo_db2=tipo_coluna_filtro,
                    inicio=inicio_teste,
                    fim=fim,
                )

                if qtd > max_linhas_recorte:
                    dia_inicio = (
                        dia_meio
                        + timedelta(days=1)
                    )

                else:
                    melhor_inicio = (
                        inicio_teste
                    )

                    melhor_qtd = qtd

                    dia_fim = (
                        dia_meio
                        - timedelta(days=1)
                    )

            if melhor_qtd is None:
                melhor_inicio = datetime.combine(
                    fim.date(),
                    time.min,
                )

                melhor_qtd = _contagem_limitada(
                    coluna=coluna_filtro,
                    tipo_db2=tipo_coluna_filtro,
                    inicio=melhor_inicio,
                    fim=fim,
                )

            truncado = (
                melhor_qtd
                > max_linhas_recorte
            )

            return (
                melhor_inicio,
                melhor_qtd,
                truncado,
            )

        # ================================================================
        # 11. Planejamento do recorte
        # ================================================================

        data_inicio_solicitada = (
            _normalizar_datetime(
                data_inicio
            )
            if data_inicio is not None
            else None
        )

        data_inicio_efetiva = None
        data_fim_efetiva = None

        where_recorte = None

        qtd_estimada_recorte = None
        recorte_truncado = False
        perfil_dados_permitido = True

        estrategia_recorte = (
            "SEM_FILTRO"
        )

        if tabela_grande:
            if (
                coluna_filtro is None
                or data_maxima is None
            ):
                perfil_dados_permitido = False

                estrategia_recorte = (
                    "SOMENTE_ESTRUTURA"
                )

                alertas.append(
                    "TABELA_GRANDE_SEM_CORTE_TEMPORAL_SEGURO"
                )

            else:
                data_fim_efetiva = (
                    data_maxima
                )

                if data_inicio_solicitada is not None:
                    inicio_minimo = (
                        data_inicio_solicitada
                    )

                    estrategia_recorte = (
                        "DATA_INICIO_INFORMADA"
                    )

                else:
                    inicio_minimo = (
                        data_maxima
                        - timedelta(
                            days=max_janela_automatica_dias
                        )
                    )

                    estrategia_recorte = (
                        "JANELA_AUTOMATICA"
                    )

                if inicio_minimo > data_maxima:
                    data_inicio_efetiva = (
                        inicio_minimo
                    )

                    qtd_estimada_recorte = 0

                else:
                    (
                        data_inicio_efetiva,
                        qtd_estimada_recorte,
                        recorte_truncado,
                    ) = _buscar_inicio_seguro(
                        inicio_minimo=inicio_minimo,
                        fim=data_maxima,
                    )

                inicio_sql = _literal_temporal(
                    data_inicio_efetiva,
                    tipo_coluna_filtro,
                )

                fim_sql = _literal_temporal(
                    data_fim_efetiva,
                    tipo_coluna_filtro,
                )

                where_recorte = (
                    f"{coluna_filtro} "
                    f">= {inicio_sql} "
                    f"AND {coluna_filtro} "
                    f"<= {fim_sql}"
                )

        else:
            # ------------------------------------------------------------
            # Tabela pequena.
            #
            # Se data_inicio foi informada e existe coluna temporal,
            # respeitamos o filtro.
            #
            # Caso contrário, podemos estudar o conjunto completo,
            # sempre mantendo FETCH FIRST como proteção.
            # ------------------------------------------------------------

            if (
                data_inicio_solicitada is not None
                and coluna_filtro is not None
                and data_maxima is not None
            ):
                data_inicio_efetiva = (
                    data_inicio_solicitada
                )

                data_fim_efetiva = (
                    data_maxima
                )

                inicio_sql = _literal_temporal(
                    data_inicio_efetiva,
                    tipo_coluna_filtro,
                )

                fim_sql = _literal_temporal(
                    data_fim_efetiva,
                    tipo_coluna_filtro,
                )

                where_recorte = (
                    f"{coluna_filtro} "
                    f">= {inicio_sql} "
                    f"AND {coluna_filtro} "
                    f"<= {fim_sql}"
                )

                estrategia_recorte = (
                    "DATA_INICIO_INFORMADA"
                )

            else:
                estrategia_recorte = (
                    "TABELA_PEQUENA_SEM_FILTRO"
                )

        # ================================================================
        # 12. Escolha automática das colunas cujo conteúdo vale estudar
        #
        # Regra principal:
        #
        # - indicador: sempre candidato;
        # - código/sigla/etc.: candidato se catálogo não mostrar
        #   cardinalidade alta;
        # - qualquer outra coluna com cardinalidade de catálogo
        #   comprovadamente pequena pode entrar;
        # - textos longos, LOB, datas, valores e números de identificação
        #   ficam fora do DISTINCT.
        # ================================================================

        colunas_para_valores = []

        for item in colunas_info:
            nome = item["coluna"]
            tipo = item["tipo_db2"]
            tamanho = item["tamanho"]
            prefixo = item[
                "prefixo_natureza"
            ]

            card = item[
                "cardinalidade_catalogo"
            ]

            if tipo in tipos_lob:
                item["diagnostico_valores"] = (
                    "NAO_APLICAVEL_LOB"
                )
                continue

            if tipo in tipos_data:
                item["diagnostico_valores"] = (
                    "NAO_APLICAVEL_TEMPORAL"
                )
                continue

            if prefixo == "VL":
                item["diagnostico_valores"] = (
                    "NAO_APLICAVEL_VALOR"
                )
                continue

            if prefixo in {
                "NR",
                "TX",
                "NM",
                "DA",
                "RI",
                "JS",
                "LS",
            }:
                item["diagnostico_valores"] = (
                    "ALTA_CARDINALIDADE_ESPERADA"
                )
                continue

            if (
                card is not None
                and card
                > limite_valores_unicos
            ):
                item["diagnostico_valores"] = (
                    "ALTA_CARDINALIDADE_CATALOGO"
                )
                continue

            candidato = False

            if prefixo == "IN":
                candidato = True

            elif prefixo in naturezas_dominio:
                candidato = True

            elif (
                card is not None
                and card
                <= limite_valores_unicos
            ):
                candidato = True

            elif (
                tipo in tipos_textuais
                and tamanho <= 8
            ):
                candidato = True

            if candidato:
                colunas_para_valores.append(
                    nome
                )

                item["diagnostico_valores"] = (
                    "CANDIDATA_DOMINIO"
                )

            else:
                item["diagnostico_valores"] = (
                    "NAO_EXPANDIR"
                )

        # ================================================================
        # 13. Materialização controlada
        #
        # Uma leitura do DB2.
        #
        # Somente colunas candidatas a domínio são trazidas.
        # ================================================================

        qtd_linhas_materializadas = 0

        if (
            perfil_dados_permitido
            and colunas_para_valores
        ):
            lista_sql = ",\n                ".join(
                colunas_para_valores
            )

            where_sql = ""

            if where_recorte:
                where_sql = (
                    f"WHERE {where_recorte}"
                )

            sql_recorte = f"""
                SELECT
                    {lista_sql}
                FROM {nome_completo}
                {where_sql}
                FETCH FIRST {max_linhas_recorte} ROWS ONLY
            """

            df_recorte = (
                self.run_select(sql_recorte)
                .cache()
            )

            try:
                qtd_linhas_materializadas = (
                    df_recorte.count()
                )

                # ========================================================
                # 14. Valores únicos somente no recorte controlado
                # ========================================================

                for nome_coluna in (
                    colunas_para_valores
                ):
                    item = coluna_por_nome[
                        nome_coluna
                    ]

                    try:
                        valores_rows = (
                            df_recorte
                            .select(
                                F.col(
                                    nome_coluna
                                ).alias("VALOR")
                            )
                            .distinct()
                            .limit(
                                limite_valores_unicos
                                + 1
                            )
                            .collect()
                        )

                        valores = [
                            _row_dict(row).get(
                                "VALOR"
                            )
                            for row
                            in valores_rows
                        ]

                        if (
                            len(valores)
                            > limite_valores_unicos
                        ):
                            item[
                                "diagnostico_valores"
                            ] = (
                                "ALTA_CARDINALIDADE_RECORTE"
                            )

                            item[
                                "valores_unicos"
                            ] = None

                        else:
                            valores_ordenados = sorted(
                                valores,
                                key=lambda x: (
                                    x is not None,
                                    str(x),
                                ),
                            )

                            item[
                                "diagnostico_valores"
                            ] = (
                                "DOMINIO_OBSERVADO"
                            )

                            item[
                                "valores_unicos"
                            ] = (
                                valores_ordenados
                            )

                    except Exception as exc:
                        item[
                            "diagnostico_valores"
                        ] = (
                            "ERRO_AO_LER_DOMINIO"
                        )

                        item[
                            "valores_unicos"
                        ] = None

                        alertas.append(
                            f"ERRO_DOMINIO_{nome_coluna}: "
                            f"{type(exc).__name__}"
                        )

            finally:
                try:
                    df_recorte.unpersist()
                except Exception:
                    pass

        # ================================================================
        # 15. Bounds seguros
        #
        # Não calculamos MIN/MAX indiscriminadamente.
        #
        # Preferimos a própria coluna temporal usada no corte.
        # ================================================================

        bounds = {}

        coluna_particao_sugerida = None

        if (
            coluna_filtro
            and coluna_filtro
            in col_particao
            and data_inicio_efetiva is not None
            and data_fim_efetiva is not None
        ):
            coluna_particao_sugerida = (
                coluna_filtro
            )

            bounds[
                coluna_particao_sugerida
            ] = {
                "lower_bound": (
                    self._normalizar_bound(
                        data_inicio_efetiva
                    )
                ),
                "upper_bound": (
                    self._normalizar_bound(
                        data_fim_efetiva
                    )
                ),
            }

        elif (
            not tabela_grande
            and col_particao
        ):
            # ------------------------------------------------------------
            # Tabela pequena:
            # podemos calcular bounds apenas da melhor candidata.
            # ------------------------------------------------------------

            coluna_particao_sugerida = (
                col_particao[0]
            )

            where_sql = ""

            if where_recorte:
                where_sql = (
                    f"WHERE {where_recorte}"
                )

            sql_bounds = f"""
                SELECT
                    MIN(
                        {coluna_particao_sugerida}
                    ) AS LOWER_BOUND,
                    MAX(
                        {coluna_particao_sugerida}
                    ) AS UPPER_BOUND
                FROM {nome_completo}
                {where_sql}
            """

            try:
                linhas_bounds = (
                    self.run_select(
                        sql_bounds
                    )
                    .collect()
                )

                if linhas_bounds:
                    dados_bounds = _row_dict(
                        linhas_bounds[0]
                    )

                    lower = dados_bounds.get(
                        "LOWER_BOUND"
                    )

                    upper = dados_bounds.get(
                        "UPPER_BOUND"
                    )

                    bounds[
                        coluna_particao_sugerida
                    ] = {
                        "lower_bound": (
                            self._normalizar_bound(
                                lower
                            )
                            if lower is not None
                            else None
                        ),
                        "upper_bound": (
                            self._normalizar_bound(
                                upper
                            )
                            if upper is not None
                            else None
                        ),
                    }

            except Exception:
                alertas.append(
                    "NAO_FOI_POSSIVEL_CALCULAR_BOUNDS"
                )

        # ================================================================
        # 16. Fetchsize
        # ================================================================

        if (
            possui_lob
            or tamanho_linha_estimado > 16_384
        ):
            fetchsize_sugerido = 1_000

        elif tamanho_linha_estimado > 4_096:
            fetchsize_sugerido = 5_000

        else:
            fetchsize_sugerido = 10_000

        # ================================================================
        # 17. Capacidade Spark/JDBC
        # ================================================================

        try:
            capacidade_spark = max(
                1,
                int(
                    self.spark
                    .sparkContext
                    .defaultParallelism
                ),
            )
        except (
            TypeError,
            ValueError,
            AttributeError,
        ):
            capacidade_spark = 1

        try:
            executor_cores = (
                self._obter_config_int(
                    "spark.executor.cores"
                )
            )
        except Exception:
            executor_cores = None

        try:
            executor_instances = (
                self._obter_config_int(
                    "spark.executor.instances"
                )
            )
        except Exception:
            executor_instances = None

        if (
            executor_cores
            and executor_instances
        ):
            capacidade_spark = max(
                capacidade_spark,
                executor_cores
                * executor_instances,
            )

        capacidade_jdbc = min(
            capacidade_spark,
            16,
        )

        # ================================================================
        # 18. Número sugerido de partições
        # ================================================================

        if coluna_particao_sugerida is None:
            num_partitions_sugerido = None

        else:
            linhas_referencia = (
                qtd_linhas_materializadas
            )

            if linhas_referencia <= 0:
                if (
                    qtd_estimada_recorte is not None
                ):
                    linhas_referencia = min(
                        qtd_estimada_recorte,
                        max_linhas_recorte,
                    )

                elif estimativa_linhas >= 0:
                    linhas_referencia = min(
                        estimativa_linhas,
                        max_linhas_recorte,
                    )

            if linhas_referencia <= 0:
                num_partitions_sugerido = 1

            elif linhas_referencia <= 250_000:
                num_partitions_sugerido = 1

            elif linhas_referencia <= 2_000_000:
                num_partitions_sugerido = min(
                    4,
                    capacidade_jdbc,
                )

            elif linhas_referencia <= 20_000_000:
                num_partitions_sugerido = min(
                    8,
                    capacidade_jdbc,
                )

            else:
                num_partitions_sugerido = (
                    capacidade_jdbc
                )

        # ================================================================
        # 19. Resultado final
        # ================================================================

        resultado = {
            "tabela": {
                "schema": schema_final,
                "nome": tabela_final,
                "nome_completo": nome_completo,
                "tipo_objeto": tipo_objeto,
                "estimativa_linhas": (
                    estimativa_linhas
                ),
                "tamanho_linha_estimado": (
                    tamanho_linha_estimado
                ),
                "possui_lob": possui_lob,
                "tabela_grande": tabela_grande,
            },

            "recorte": {
                "estrategia": (
                    estrategia_recorte
                ),
                "perfil_dados_permitido": (
                    perfil_dados_permitido
                ),
                "coluna_filtro": (
                    coluna_filtro
                ),
                "tipo_coluna_filtro": (
                    tipo_coluna_filtro
                ),
                "data_inicio_solicitada": (
                    data_inicio_solicitada
                ),
                "data_inicio_efetiva": (
                    data_inicio_efetiva
                ),
                "data_fim_efetiva": (
                    data_fim_efetiva
                ),
                "where": where_recorte,
                "qtd_estimada_limitada": (
                    qtd_estimada_recorte
                ),
                "qtd_materializada": (
                    qtd_linhas_materializadas
                ),
                "recorte_truncado": (
                    recorte_truncado
                ),
                "max_linhas_recorte": (
                    max_linhas_recorte
                ),
            },

            "candidatas_temporais": (
                candidatas_temporais
            ),

            "colunas": colunas_info,

            "colunas_valores_unicos": [
                item["coluna"]
                for item in colunas_info
                if item["diagnostico_valores"]
                == "DOMINIO_OBSERVADO"
            ],

            # ------------------------------------------------------------
            # Compatibilidade com o método anterior
            # ------------------------------------------------------------

            "filtrar_cols": filtrar_cols,

            "col_data": col_data,

            "col_particao": (
                col_particao
            ),

            "col_particao_sugerida": (
                coluna_particao_sugerida
            ),

            "bounds": bounds,

            "fetchsize": (
                fetchsize_sugerido
            ),

            "num_partitions": (
                num_partitions_sugerido
            ),

            "alertas": alertas,
        }

        # ================================================================
        # 20. Exibição
        # ================================================================

        if show:
            print("=" * 88)
            print(
                "DIAGNÓSTICO AUTOMÁTICO DB2"
            )
            print("=" * 88)

            print(
                f"Tabela: {nome_completo}"
            )

            print(
                f"Tipo objeto: {tipo_objeto}"
            )

            if estimativa_linhas >= 0:
                print(
                    "Estimativa de linhas: "
                    f"{estimativa_linhas:,.0f}"
                )
            else:
                print(
                    "Estimativa de linhas: "
                    "indisponível"
                )

            print(
                "Tabela classificada como grande: "
                f"{tabela_grande}"
            )

            print(
                "Tamanho estimado da linha: "
                f"{tamanho_linha_estimado} bytes"
            )

            print("\n" + "-" * 88)
            print("CORTE AUTOMÁTICO")
            print("-" * 88)

            if candidatas_temporais:
                print(
                    "Candidatas temporais:"
                )

                for candidata in (
                    candidatas_temporais
                ):
                    print(
                        f"  "
                        f"{candidata['coluna']}: "
                        f"score="
                        f"{candidata['score']}, "
                        f"indice_lider="
                        f"{candidata['indice_lider']}"
                    )

            else:
                print(
                    "Nenhuma coluna temporal encontrada."
                )

            print(
                "\nColuna escolhida: "
                f"{coluna_filtro}"
            )

            print(
                "Estratégia: "
                f"{estrategia_recorte}"
            )

            print(
                "Perfil de dados permitido: "
                f"{perfil_dados_permitido}"
            )

            if data_inicio_solicitada is not None:
                print(
                    "Data início solicitada: "
                    f"{data_inicio_solicitada}"
                )

            if data_inicio_efetiva is not None:
                print(
                    "Data início efetiva: "
                    f"{data_inicio_efetiva}"
                )

            if data_fim_efetiva is not None:
                print(
                    "Data fim efetiva: "
                    f"{data_fim_efetiva}"
                )

            if where_recorte:
                print(
                    "Predicado efetivo:"
                )
                print(
                    f"  {where_recorte}"
                )

            print(
                "Máximo de linhas do recorte: "
                f"{max_linhas_recorte:,}"
            )

            if qtd_linhas_materializadas:
                print(
                    "Linhas materializadas: "
                    f"{qtd_linhas_materializadas:,}"
                )

            print("\n" + "-" * 88)
            print("DIAGNÓSTICO DAS COLUNAS")
            print("-" * 88)

            linhas_exibicao = []

            for item in colunas_info:
                valores = item[
                    "valores_unicos"
                ]

                if valores is None:
                    valores_texto = ""
                else:
                    valores_texto = ", ".join(
                        "NULL"
                        if valor is None
                        else str(valor)
                        for valor in valores
                    )

                linhas_exibicao.append(
                    (
                        item["posicao"],
                        item["coluna"],
                        item["natureza"],
                        item["tipo_db2"],
                        item["tamanho"],
                        item["aceita_nulo"],
                        item["indice_lider"],
                        item[
                            "diagnostico_valores"
                        ],
                        valores_texto,
                    )
                )

            df_diagnostico = (
                self.spark.createDataFrame(
                    linhas_exibicao,
                    [
                        "POS",
                        "COLUNA",
                        "NATUREZA",
                        "TIPO_DB2",
                        "TAMANHO",
                        "NULLABLE",
                        "INDICE_LIDER",
                        "DIAGNOSTICO",
                        "VALORES_UNICOS_OBSERVADOS",
                    ],
                )
            )

            df_diagnostico.show(
                n=len(linhas_exibicao),
                truncate=truncate,
            )

            print("\n" + "-" * 88)
            print(
                "INSUMOS PARA selecionar_tabela"
            )
            print("-" * 88)

            print("\nfiltrar_cols =")
            print(
                self._formatar_lista_copiavel(
                    filtrar_cols
                )
            )

            print("\ncol_data =")
            print(
                self._formatar_lista_copiavel(
                    col_data
                )
            )

            print("\ncol_particao =")
            print(
                self._formatar_lista_copiavel(
                    col_particao
                )
            )

            print(
                "\ncol_particao_sugerida = "
                f"{coluna_particao_sugerida}"
            )

            print("\nbounds =")

            if bounds:
                for coluna, valores in (
                    bounds.items()
                ):
                    print(
                        f"  {coluna}: "
                        f"{valores}"
                    )
            else:
                print("  {}")

            print(
                f"\nfetchsize = "
                f"{fetchsize_sugerido}"
            )

            print(
                "num_partitions = "
                f"{num_partitions_sugerido}"
            )

            if alertas:
                print("\n" + "-" * 88)
                print("ALERTAS")
                print("-" * 88)

                for alerta in alertas:
                    print(
                        f"- {alerta}"
                    )

            print("=" * 88)

        return resultado

    def selecionar_tabela(
        self,
        schema: str,
        nome_tabela: str,
        fetchsize: Optional[int] = None,
        partition_column: Optional[str] = None,
        lower_bound: Optional[int] = None,
        upper_bound: Optional[int] = None,
        num_partitions: Optional[int] = None,
    ) -> DataFrame:
        schema_final = (schema or "").strip().upper()
        tabela_final = (nome_tabela or "").strip().upper()

        if not schema_final:
            raise ValueError("schema nao pode ser vazio")

        if not tabela_final:
            raise ValueError("nome_tabela nao pode ser vazio")

        if "." in tabela_final:
            raise ValueError(
                "nome_tabela deve receber apenas o nome da tabela. "
                "Informe o schema separadamente."
            )

        return self.run_select(
            sql=f"SELECT * FROM {schema_final}.{tabela_final}",
            fetchsize=fetchsize,
            partition_column=partition_column,
            lower_bound=lower_bound,
            upper_bound=upper_bound,
            num_partitions=num_partitions,
        )


def criar_cliente_db2_spark(env: Optional[Dict[str, str]] = None) -> ClientDb2Spark:
    return ClientDb2Spark(spark=spark, env=env)


def help_gerenciador_sessao_spark_remoto() -> None:
    print("""
============================================================
gerenciador_sessao_spark_remoto.ipynb
============================================================

Este notebook deve ser executado depois da criacao da sessao Spark.

Ele disponibiliza na sessao Spark:

- ler_variavel_ambiente_spark
- criar_logger_spark
- logger
- ClientOracleSpark
- criar_cliente_oracle_spark
- ClientDb2Spark
- criar_cliente_db2_spark

Leituras Oracle e DB2 usam spark.read.format("jdbc").options(...).load().
Escritas Oracle usam writer JDBC Spark.
Comandos Oracle sem retorno usam conexao JDBC direta.

============================================================
""")
